## Sel 1: Import Pustaka

In [5]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

import mlflow
import mlflow.sklearn
import mlflow.xgboost

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
import xgboost as xgb

from sklearn.multioutput import MultiOutputRegressor

import joblib
import contextlib
from tqdm.auto import tqdm

import warnings
warnings.filterwarnings('ignore')

## Sel 2: Memuat & Pra-pemrosesan Data (Resampling Harian)

In [6]:
# 1. BACA CREDENTIAL DARI .ENV
load_dotenv()
db_url = os.getenv("DATABASE_URL")

# Supabase kadang memberikan URL dengan awalan 'postgres://'
# Sedangkan SQLAlchemy mewajibkan 'postgresql://'. Kita atasi di sini:
if db_url and db_url.startswith("postgres://"):
    db_url = db_url.replace("postgres://", "postgresql://", 1)

if not db_url:
    raise ValueError("🚨 DATABASE_URL tidak ditemukan! Pastikan file .env sudah ada.")

# 2. BUAT KONEKSI KE DATABASE
print("Membuka jembatan ke Supabase...")
engine = create_engine(db_url)

# 3. KUERI SQL (Mengambil data historis)
# Kita ambil id_wilayah sebagai referensi utama, bukan teks nama kota
query_sql = """
    SELECT 
        waktu_aktual, 
        id_wilayah, 
        pm25, 
        pm10, 
        so2, 
        co, 
        no2, 
        ozon
    FROM data_historis
    ORDER BY waktu_aktual ASC;
"""

print("Menyedot data dari pangkalan data. Mohon tunggu...")
df = pd.read_sql(query_sql, engine)

# --- BAGIAN INI SUDAH DISESUAIKAN DENGAN SKEMA SUPABASE ---
kolom_waktu = 'waktu_aktual'
kolom_kota = 'id_wilayah'  # Sekarang menggunakan ID, bukan nama teks
polutan = ['pm25', 'pm10', 'so2', 'co', 'no2', 'ozon'] # Nama polutan versi ringkas
# -----------------------------------------------------------

# 4. PRA-PEMROSESAN FORMAT WAKTU
df[kolom_waktu] = pd.to_datetime(df[kolom_waktu])

# Karena waktu dari database mungkin memiliki info zona waktu (tz-aware),
# kita seragamkan menjadi tz-naive agar Pandas tidak kebingungan saat resample
df[kolom_waktu] = df[kolom_waktu].dt.tz_localize(None)

df.set_index(kolom_waktu, inplace=True)

# 5. RESAMPLE HARIAN BERDASARKAN ID WILAYAH
# Pastikan kolom_kota dan polutan di-subset dengan benar
df_daily = df.groupby(kolom_kota)[polutan].resample('D').mean().reset_index()

# Hapus baris kosong
df_daily = df_daily.dropna()

print(f"✅ Selesai! Bentuk data setelah dijadikan rata-rata harian: {df_daily.shape}")
display(df_daily.head())

Membuka jembatan ke Supabase...
Menyedot data dari pangkalan data. Mohon tunggu...
✅ Selesai! Bentuk data setelah dijadikan rata-rata harian: (1140, 8)


,id_wilayah,waktu_aktual,pm25,pm10,so2,co,no2,ozon
0,1,2026-04-15,6.629231,9.167692,0.716923,150.942308,1.432308,45.881538
1,1,2026-04-16,10.141250,16.392917,1.060833,235.865000,2.859583,47.792083
2,1,2026-04-17,5.637500,10.565417,0.632083,148.017083,1.315000,46.709167
3,1,2026-04-18,4.465833,8.722917,0.355833,111.362083,0.932500,38.266250
4,1,2026-04-19,4.371250,10.059167,0.407083,122.011667,1.260000,44.565000


## Sel 3: Rekayasa Fitur (Sliding Window 3 Hari & Target Besok)

In [7]:
# Fungsi untuk membuat Sliding Window, Fitur Temporal, dan Rolling Stats
def create_advanced_features(df_kota, n_lags=3):
    df_temp = df_kota.copy()
    
    # 🌟 FITUR BARU 1: KONTEKS WAKTU (TEMPORAL)
    df_temp['Bulan'] = df_temp[kolom_waktu].dt.month
    df_temp['Is_Weekend'] = df_temp[kolom_waktu].dt.dayofweek.isin([5, 6]).astype(int)

    for p in polutan:
        # --- FITUR LAMA: HISTORY (t-1, t-2, t-3) ---
        for i in range(1, n_lags + 1):
            df_temp[f'{p}_H-{i}'] = df_temp[p].shift(i)
            
        # 🌟 FITUR BARU 2: TREN JANGKA MENENGAH (ROLLING STATS)
        df_temp[f'{p}_RollMean_3'] = df_temp[p].shift(1).rolling(window=3).mean()
        df_temp[f'{p}_RollMax_3'] = df_temp[p].shift(1).rolling(window=3).max()
        
        # --- TARGET BESOK (t+1) ---
        df_temp[f'TARGET_{p}_Besok'] = df_temp[p].shift(-1)
        
    return df_temp

print("Meracik fitur tingkat lanjut...")

# Aplikasikan fungsi di atas untuk masing-masing kota/wilayah secara terpisah
df_model = df_daily.groupby(kolom_kota, group_keys=False).apply(create_advanced_features)

# Hapus baris yang memiliki NaN akibat pergeseran history dan rolling
df_model = df_model.dropna().reset_index(drop=True)

# Lakukan One-Hot Encoding untuk kolom id_wilayah (kolom_kota)
# Ubah ke string dulu agar Pandas tahu ini adalah kategori, bukan angka ukur
df_model[kolom_kota] = df_model[kolom_kota].astype(str)
df_model = pd.get_dummies(df_model, columns=[kolom_kota])

# 1. Kumpulkan semua nama kolom target
kolom_target = [col for col in df_model.columns if 'TARGET_' in col]

# 2. Pisahkan tabel jawaban (y)
y = df_model[kolom_target]

# 3. Pisahkan tabel soal (X) 
X = df_model.drop(columns=kolom_target + [kolom_waktu])

print(f"Bentuk tabel awal df_model : {df_model.shape}")
print(f"Dimensi X (Fitur AI BARU!) : {X.shape}")
print(f"Dimensi y (Target AI)      : {y.shape}")

Meracik fitur tingkat lanjut...
Bentuk tabel awal df_model : (988, 83)
Dimensi X (Fitur AI BARU!) : (988, 76)
Dimensi y (Target AI)      : (988, 6)


## Sel 4: Splitting & Training Baseline Model MLFLOW

In [8]:
# 1. Splitting Data (80% Training, 20% Testing) 
# 🚨 PERBAIKAN KRUSIAL: shuffle=False agar tidak terjadi Time Leakage!
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# Set nama eksperimen utama di MLflow
mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("Proyek_ISPU_Jatim")

# 2. Persiapan 3 Baseline Model
baselines = {
    "Baseline_LinearRegression": MultiOutputRegressor(LinearRegression()),
    "Baseline_RandomForest": MultiOutputRegressor(RandomForestRegressor(n_estimators=50, random_state=42)),
    "Baseline_KNeighbors": MultiOutputRegressor(KNeighborsRegressor())
}

print("--- MEMULAI TRACKING BASELINE MODELS DI MLFLOW ---")
for nama_model, model in baselines.items():
    # Membuka sesi pencatatan MLflow untuk setiap model
    with mlflow.start_run(run_name=nama_model):
        
        # Training & Prediksi
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        # Kalkulasi Metrik Keseluruhan
        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)
        
        # Mencatat parameter, metrik, dan artifact model ke MLflow
        mlflow.log_param("algoritma", nama_model)
        mlflow.log_metric("test_mae", mae)
        mlflow.log_metric("test_rmse", rmse)
        mlflow.log_metric("test_r2", r2)
        mlflow.sklearn.log_model(model, artifact_path=nama_model)
        
        print(f"✅ {nama_model} berhasil dicatat (RMSE: {rmse:.2f})")

Traceback (most recent call last):
  File "c:\Users\THIN 15\AppData\Local\Programs\Python\Python312\Lib\site-packages\mlflow\store\tracking\file_store.py", line 383, in search_experiments
    exp = self._get_experiment(exp_id, view_type)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\THIN 15\AppData\Local\Programs\Python\Python312\Lib\site-packages\mlflow\store\tracking\file_store.py", line 481, in _get_experiment
    meta = FileStore._read_yaml(experiment_dir, FileStore.META_DATA_FILE_NAME)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\THIN 15\AppData\Local\Programs\Python\Python312\Lib\site-packages\mlflow\store\tracking\file_store.py", line 1670, in _read_yaml
    return _read_helper(root, file_name, attempts_remaining=retries)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\THIN 15\AppData\Local\Programs\Python\Python312\Lib\site-packages\mlflow\store\tracking\file_store.py",

--- MEMULAI TRACKING BASELINE MODELS DI MLFLOW ---


2026/05/14 16:52:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/14 16:52:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


✅ Baseline_LinearRegression berhasil dicatat (RMSE: 30.76)


2026/05/14 16:52:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/14 16:52:57 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


✅ Baseline_RandomForest berhasil dicatat (RMSE: 14.74)


2026/05/14 16:53:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/14 16:53:04 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


✅ Baseline_KNeighbors berhasil dicatat (RMSE: 21.78)


## Sel 5 : Hyperparameter Tuning XGBoost MLFlow

In [9]:
print("Memulai Hyperparameter Tuning XGBoost (Strategi Pisahkan Otak)...")

# 1. Siapkan 'brankas' untuk menyimpan 6 otak AI yang berbeda
model_terbaik_per_polutan = {}

# 2. Ruang Pencarian Grid
param_grid_murni = {
    'n_estimators': [100, 200, 300, 400],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 5, 7, 9],
    'subsample': [0.8, 0.9, 1.0],
    'colsample_bytree': [0.8, 0.9, 1.0]
}

# --- JEMBATAN TQDM & JOBLIB ---
@contextlib.contextmanager
def tqdm_joblib(tqdm_object):
    class TqdmBatchCompletionCallback(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)
    old_batch_callback = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback
    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old_batch_callback
# ------------------------------------------

total_mae, total_rmse = 0, 0
daftar_kolom_target = y.columns 

# 3. Lakukan Perulangan: Latih 1 AI untuk tiap 1 Polutan
for nama_kolom in daftar_kolom_target:
    
    # MEMBERSIHKAN NAMA UNTUK MLFLOW & PRINT (Kini jauh lebih simpel!)
    # Mengubah 'TARGET_pm25_Besok' menjadi 'pm25'
    nama_polutan_bersih = nama_kolom.replace('TARGET_', '').replace('_Besok', '')
    
    print(f"\n======================================================")
    print(f"🚀 Mendidik AI Spesialis Khusus: {nama_polutan_bersih.upper()}")
    print(f"======================================================")
    
    # Ambil kolom jawaban (y) HANYA untuk gas ini
    y_train_tunggal = y_train[nama_kolom]
    y_test_tunggal = y_test[nama_kolom]
    
    # Inisiasi XGBoost Murni
    model_xgb_dasar = xgb.XGBRegressor(random_state=42)
    
    # Siapkan Mesin Pencari
    grid_search = RandomizedSearchCV(
        estimator=model_xgb_dasar, 
        param_distributions=param_grid_murni, 
        n_iter=15, 
        cv=4,      
        scoring='neg_mean_absolute_error', 
        random_state=42,
        n_jobs=-1, 
        verbose=0  
    )
    
    # [MLFLOW] Membuka Sesi Pencatatan untuk Polutan Spesifik
    with mlflow.start_run(run_name=f"Spesialis_XGBoost_{nama_polutan_bersih.upper()}"):
        
        total_fits = grid_search.n_iter * grid_search.cv 
        with tqdm_joblib(tqdm(desc=f"Tuning {nama_polutan_bersih.upper()}", total=total_fits, unit="fit")):
            grid_search.fit(X_train, y_train_tunggal)
            
        # Ambil otak terbaiknya
        otak_spesialis = grid_search.best_estimator_
        
        # Simpan ke dalam brankas dictionary
        model_terbaik_per_polutan[nama_kolom] = otak_spesialis
        
        print(f"Kombinasi Terbaik:\n{grid_search.best_params_}")
        
        # Lakukan ujian evaluasi langsung
        y_pred_tunggal = otak_spesialis.predict(X_test)
        
        mae = mean_absolute_error(y_test_tunggal, y_pred_tunggal)
        rmse = np.sqrt(mean_squared_error(y_test_tunggal, y_pred_tunggal))
        rata_rata_asli = y_test_tunggal.mean()
        persentase_error = (mae / rata_rata_asli) * 100 if rata_rata_asli > 0 else 0
        
        total_mae += mae
        total_rmse += rmse
        
        print(f"  - Rata-rata asli : {rata_rata_asli:.2f}")
        print(f"  - MAE (Meleset)  : {mae:.2f}")
        print(f"  - RMSE           : {rmse:.2f}")
        print(f"  - Error (%)      : {persentase_error:.2f}%")
        
        # [MLFLOW] Catat skor evaluasi ke dashboard
        mlflow.log_metric("test_mae", mae)
        mlflow.log_metric("test_rmse", rmse)
        
        # [MLFLOW] Catat resep parameter yang menang ke dashboard
        for param_name, param_val in grid_search.best_params_.items():
            mlflow.log_param(param_name, param_val)
            
        # [MLFLOW] Simpan model spesialis ini ke brankas MLflow
        mlflow.sklearn.log_model(otak_spesialis, artifact_path=f"Model_{nama_polutan_bersih.upper()}")

print("\n--- 🏁 SEMUA 6 AI SPESIALIS SELESAI DILATIH ---")
print(f"Rata-rata MAE Keseluruhan : {total_mae/6:.2f}")

Memulai Hyperparameter Tuning XGBoost (Strategi Pisahkan Otak)...

🚀 Mendidik AI Spesialis Khusus: PM25


Tuning PM25:   0%|          | 0/60 [00:00<?, ?fit/s]

2026/05/14 16:53:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/14 16:53:31 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Kombinasi Terbaik:
{'subsample': 0.8, 'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.05, 'colsample_bytree': 1.0}
  - Rata-rata asli : 4.03
  - MAE (Meleset)  : 1.28
  - RMSE           : 2.30
  - Error (%)      : 31.89%

🚀 Mendidik AI Spesialis Khusus: PM10


Tuning PM10:   0%|          | 0/60 [00:00<?, ?fit/s]

...

2026/05/14 16:55:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/14 16:55:08 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Kombinasi Terbaik:
{'subsample': 0.8, 'n_estimators': 300, 'max_depth': 5, 'learning_rate': 0.1, 'colsample_bytree': 1.0}
  - Rata-rata asli : 0.35
  - MAE (Meleset)  : 0.24
  - RMSE           : 0.51
  - Error (%)      : 66.95%

🚀 Mendidik AI Spesialis Khusus: OZON


Tuning OZON:   0%|          | 0/60 [00:00<?, ?fit/s]

2026/05/14 16:55:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/14 16:55:34 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Kombinasi Terbaik:
{'subsample': 0.9, 'n_estimators': 400, 'max_depth': 7, 'learning_rate': 0.05, 'colsample_bytree': 0.8}
  - Rata-rata asli : 36.04
  - MAE (Meleset)  : 2.42
  - RMSE           : 3.58
  - Error (%)      : 6.73%

--- 🏁 SEMUA 6 AI SPESIALIS SELESAI DILATIH ---
Rata-rata MAE Keseluruhan : 3.38


## Sel 6: Ekspor Model (Menyimpan Hasil)

In [10]:
os.makedirs('data_training', exist_ok=True)

# Membungkus KE-ENAM model spesialis dan daftar nama fitur
paket_model = {
    'dict_model_spesialis': model_terbaik_per_polutan, # Berisi 6 model XGBoost
    'fitur': list(X_train.columns)
}

# Menyimpan ke dalam file Pickle langsung ke kandang DVC
nama_file = 'data_training/xgb_ispu_jatim_multi_otak.pkl'
joblib.dump(paket_model, nama_file)

print(f"Selesai! 6 Model pintar telah dibungkus jadi satu dan diekspor sebagai '{nama_file}'")

# [MLFLOW] Mencatat file bundle pkl ke dalam run khusus
with mlflow.start_run(run_name="Final_Bundle_Export"):
    mlflow.log_artifact(nama_file)
    print(f"📦 File {nama_file} juga telah diunggah ke MLflow Artifacts.")

Selesai! 6 Model pintar telah dibungkus jadi satu dan diekspor sebagai 'data_training/xgb_ispu_jatim_multi_otak.pkl'
📦 File data_training/xgb_ispu_jatim_multi_otak.pkl juga telah diunggah ke MLflow Artifacts.
